In [ ]:
!pip install qiskit

In [ ]:
# %% [markdown]
# # Quantum Track: Parameterized Quantum Circuit (PQC)
# File: `quantum_parameterized_circuit.ipynb`
# Compatible with Qiskit 1.0+
#
# Pipeline:
# $$|0\rangle \longrightarrow R_z(\theta_1) \longrightarrow R_y(\theta_2) \longrightarrow \text{Measurement } \langle Z \rangle \longrightarrow \text{Loss Function} \longrightarrow \text{Gradient Update}$$

# %%
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

# Set random seed for reproducibility
np.random.seed(42)

# %% [markdown]
# ## 1. Build Parameterized Quantum Circuit (PQC)

# %%
def build_pqc(theta1: float, theta2: float) -> QuantumCircuit:
    """
    Constructs a 1-qubit parameterized quantum circuit:
    |0> -> Rz(theta1) -> Ry(theta2) -> Measurement
    """
    qc = QuantumCircuit(1)
    
    # Unitary U(theta1): Rz rotation
    qc.rz(theta1, 0)
    
    # Unitary U(theta2): Ry rotation
    qc.ry(theta2, 0)
    
    return qc

# Visualize the circuit with sample parameters
sample_qc = build_pqc(theta1=np.pi/4, theta2=np.pi/3)
print("Circuit Diagram:")
print(sample_qc.draw(output='text'))

# %% [markdown]
# ## 2. Measure Expectation Value $\langle Z \rangle$ using Qiskit 1.0+ StatevectorEstimator

# %%
# Define Observable Z
observable_Z = SparsePauliOp.from_list([("Z", 1.0)])

# Instantiate Qiskit 1.0+ Primitive
estimator = StatevectorEstimator()

def compute_expectation_value(theta1: float, theta2: float) -> float:
    """
    Computes <Z> using Qiskit 1.0+ StatevectorEstimator Primitive.
    """
    qc = build_pqc(theta1, theta2)
    
    # Format PUB (Primitive Unified Block): (circuit, observable)
    pub = (qc, observable_Z)
    
    # Run the estimator primitive job
    job = estimator.run([pub])
    result = job.result()[0]
    
    # Extract expectation value array
    expectation = float(result.data.evs)
    return expectation

# Test expectation value
exp_val = compute_expectation_value(theta1=0.0, theta2=np.pi)
print(f"Expectation value <Z> for Ry(pi): {exp_val:.4f} (Expected: -1.0)")

# %% [markdown]
# ## 3. Parameter-Shift Rule for Exact Analytical Gradients

# %%
def compute_gradients_parameter_shift(theta1: float, theta2: float) -> tuple[float, float]:
    """
    Calculates exact gradients d<Z>/d(theta1) and d<Z>/d(theta2) 
    using the Parameter-Shift Rule: d<Z>/d(theta) = 0.5 * (<Z>(theta + pi/2) - <Z>(theta - pi/2))
    """
    shift = np.pi / 2
    
    # Gradient w.r.t theta1
    exp_t1_plus = compute_expectation_value(theta1 + shift, theta2)
    exp_t1_minus = compute_expectation_value(theta1 - shift, theta2)
    grad_theta1 = 0.5 * (exp_t1_plus - exp_t1_minus)
    
    # Gradient w.r.t theta2
    exp_t2_plus = compute_expectation_value(theta1, theta2 + shift)
    exp_t2_minus = compute_expectation_value(theta1, theta2 - shift)
    grad_theta2 = 0.5 * (exp_t2_plus - exp_t2_minus)
    
    return grad_theta1, grad_theta2

# %% [markdown]
# ## 4. Define Loss Function and Optimization Loop
# Target: Minimize Loss $L(\theta_1, \theta_2) = \langle Z \rangle$ towards minimum value $-1.0$ (State $|1\rangle$).

# %%
# Initialize parameters randomly
theta1 = float(np.random.uniform(0, 2 * np.pi))
theta2 = float(np.random.uniform(0, 2 * np.pi))

learning_rate = 0.1
iterations = 50

history_loss = []
history_theta1 = []
history_theta2 = []

print(f"Initial parameters: theta1 = {theta1:.4f}, theta2 = {theta2:.4f}")

for step in range(iterations):
    # 1. Forward Pass: Compute Expectation / Loss
    exp_val = compute_expectation_value(theta1, theta2)
    loss = exp_val  # Direct expectation optimization target
    
    history_loss.append(loss)
    history_theta1.append(theta1)
    history_theta2.append(theta2)
    
    # 2. Backward Pass: Parameter-Shift Gradients
    grad_t1, grad_t2 = compute_gradients_parameter_shift(theta1, theta2)
    
    # 3. Parameter Updates
    theta1 -= learning_rate * grad_t1
    theta2 -= learning_rate * grad_t2

    if (step + 1) % 10 == 0 or step == 0:
        print(f"Step [{step+1:2d}/{iterations}] | Loss <Z>: {loss:8.4f} | grad_t1: {grad_t1:7.4f} | grad_t2: {grad_t2:7.4f}")

print(f"\nFinal Optimized Parameters: theta1 = {theta1:.4f}, theta2 = {theta2:.4f}")
print(f"Final Minimum Expectation Value <Z>: {compute_expectation_value(theta1, theta2):.4f}")

# %% [markdown]
# ## 5. Visualization: Loss vs Iterations

# %%
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, iterations + 1), history_loss, color='purple', linewidth=2)
plt.axhline(-1.0, linestyle='--', color='red', alpha=0.7, label='Theoretical Min (-1.0)')
plt.title("PQC Optimization: Loss $\langle Z \\rangle$ vs Iteration", fontweight='bold')
plt.xlabel("Iteration")
plt.ylabel("Loss / Expectation Value $\langle Z \\rangle$")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_theta1, label=r"$\theta_1$ (Rz)", linewidth=2)
plt.plot(history_theta2, label=r"$\theta_2$ (Ry)", linewidth=2)
plt.title("Parameter Trajectories over Optimization", fontweight='bold')
plt.xlabel("Iteration")
plt.ylabel("Angle (Radians)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()